## Dataset

> The dataset for this homework is derived from the text8 collection, which comes from Wikipedia. Your method will use character-level tokenization and operate over text8 sequences that are each exactly 20 characters long. Only 27 character types are present (lowercase characters and spaces); special characters are replaced by a single space and numbers are spelled out as individual digits (50 becomes five zero). Part of examples are:
> 
> - heir average albedo
> - ed by rank and file
> - s can also extend in
> - erages between nine
> - that civilization n
> - on a t shaped islan
> 
> The dataset is in lettercounting-train.txt and lettercounting-dev.txt. Both two files contain character strings of length 20. You can assume that your model will always see 20 characters as input and make a prediction at each position in the sequence.

## Code

> The framework code you are given consists of several files.
> 1. *utils.py*: it implements an Indexer class, which can be used to maintain a bijective mapping between indices and features (strings).
> 2. *letter_counting.py*: contains the driver code, which imports transformer.py, the file you will be editing for this assignment.
> 3. *transformer.py*: **You need to fill out all missing parts. Note that your solutions should not use nn.TransformerEncoder, nn.TransformerDecoder, or any other off-the-shelf self-attention layers. You can use nn.Linear, nn.Embedding, and PyTorch’s provided nonlinearities / loss functions to implement Transformers from scratch.**


In [1]:
!python letter_counting.py --task BEFORE

Namespace(task='BEFORE', train='data/lettercounting-train.txt', dev='data/lettercounting-dev.txt', output_bundle_path='classifier-output.json')
['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ']
10000 lines read in
1000 lines read in
Model architecture:
Transformer(
  (tok_emb): Embedding(27, 64)
  (pos_enc): PositionalEncoding(
    (emb): Embedding(20, 64)
  )
  (layers): ModuleList(
    (0-1): 2 x TransformerLayer(
      (W_q): Linear(in_features=64, out_features=64, bias=True)
      (W_k): Linear(in_features=64, out_features=64, bias=True)
      (W_v): Linear(in_features=64, out_features=64, bias=True)
      (W_o): Linear(in_features=64, out_features=64, bias=True)
      (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=64, out_features=256, bias=True)
        (gelu): GE

## BEFORE task — Implementation summary

**Model** (`transformer.py`):

- Character embedding (27 → 64) + learned positional embedding (20 → 64)
- 2 × `TransformerLayer`:
  - Pre-LayerNorm → **hand-written single-head self-attention** (Q/K/V via `nn.Linear`, `softmax(QKᵀ/√d)`, reused the template's `MLP` as FFN)
  - Causal mask (upper-triangular → -inf) so each position only attends to previous positions
- Final LayerNorm → `Linear(64 → 3)` → `log_softmax`
- Trained with `Adam(lr=1e-3)` + `NLLLoss`, batch size 64, 10 epochs

**Observed output (summary)**

- Per-epoch dev accuracy (200 exs): rises from 0.78 → 0.99 over 10 epochs (~6 s / epoch on CPU).
- First 5 dev examples: **100 / 100 positions correct** (GOLD == PRED for all 5 samples).
- Training accuracy on 100 training examples: **99.25%**.
- **Dev accuracy (whole 1000-example set): 99.50%**.

The full per-epoch log and decode output for the BEFORE task are produced by the cell above.

### Attention visualizations — BEFORE

Each image is the attention map of one layer on one dev example (rows = query position, cols = key position).
Because the task is causal, attention is concentrated in the lower triangle — each position only attends to earlier characters,
and the model learns to attend to previous occurrences of the same character in order to count them.

**Example 0 — `heir average albedo `**

Layer 0 | Layer 1
:---:|:---:
![](plots_before/0_attns0.png) | ![](plots_before/0_attns1.png)

**Example 1 — `ed by rank and file `**

Layer 0 | Layer 1
:---:|:---:
![](plots_before/1_attns0.png) | ![](plots_before/1_attns1.png)

**Example 2 — `s can also extend in`**

Layer 0 | Layer 1
:---:|:---:
![](plots_before/2_attns0.png) | ![](plots_before/2_attns1.png)

**Example 3 — `erages between nine `**

Layer 0 | Layer 1
:---:|:---:
![](plots_before/3_attns0.png) | ![](plots_before/3_attns1.png)

**Example 4 — ` that civilization n`**

Layer 0 | Layer 1
:---:|:---:
![](plots_before/4_attns0.png) | ![](plots_before/4_attns1.png)

In [2]:
!python letter_counting.py --task BEFOREAFTER 

Namespace(task='BEFOREAFTER', train='data/lettercounting-train.txt', dev='data/lettercounting-dev.txt', output_bundle_path='classifier-output.json')
['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ']
10000 lines read in
1000 lines read in
Model architecture:
Transformer(
  (tok_emb): Embedding(27, 64)
  (pos_enc): PositionalEncoding(
    (emb): Embedding(20, 64)
  )
  (layers): ModuleList(
    (0-1): 2 x TransformerLayer(
      (W_q): Linear(in_features=64, out_features=64, bias=True)
      (W_k): Linear(in_features=64, out_features=64, bias=True)
      (W_v): Linear(in_features=64, out_features=64, bias=True)
      (W_o): Linear(in_features=64, out_features=64, bias=True)
      (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=64, out_features=256, bias=True)
        (gelu

## BEFOREAFTER task — Implementation summary

Same architecture as BEFORE, but the `TransformerLayer` is instantiated with `causal=False`
so that every position can attend to every other position (the task requires counting all other occurrences,
not only the previous ones). The attention mask is simply not applied.

**Observed output (summary)**

- Per-epoch dev accuracy (200 exs): rises from 0.83 → 0.99 over 10 epochs (~5 s / epoch on CPU).
- First 5 dev examples: **98 / 100 positions correct** (4 of 5 samples fully correct, one sample has 2 wrong positions).
- Training accuracy on 100 training examples: **98.55%**.
- **Dev accuracy (whole 1000-example set): 98.52%**.

### Attention visualizations — BEFOREAFTER

Attention is no longer restricted to the lower triangle — each position can look at the whole sequence,
and the model learns to attend to the other occurrences of the same character (ignoring itself) to count them.

**Example 0 — `heir average albedo `**

Layer 0 | Layer 1
:---:|:---:
![](plots_beforeafter/0_attns0.png) | ![](plots_beforeafter/0_attns1.png)

**Example 1 — `ed by rank and file `**

Layer 0 | Layer 1
:---:|:---:
![](plots_beforeafter/1_attns0.png) | ![](plots_beforeafter/1_attns1.png)

**Example 2 — `s can also extend in`**

Layer 0 | Layer 1
:---:|:---:
![](plots_beforeafter/2_attns0.png) | ![](plots_beforeafter/2_attns1.png)

**Example 3 — `erages between nine `**

Layer 0 | Layer 1
:---:|:---:
![](plots_beforeafter/3_attns0.png) | ![](plots_beforeafter/3_attns1.png)

**Example 4 — ` that civilization n`**

Layer 0 | Layer 1
:---:|:---:
![](plots_beforeafter/4_attns0.png) | ![](plots_beforeafter/4_attns1.png)

- This assignment was mainly adopted from [link](https://www.cs.utexas.edu/~gdurrett/courses/online-course/a3.pdf).

**Upload your homework: use html file format.**

In [3]:
!jupyter nbconvert assignment-02-24300980021-刘俊彦.ipynb --to html --template classic --embed-images

[NbConvertApp] Converting notebook assignment-02-24300980021-刘俊彦.ipynb to html
[NbConvertApp] Writing 544522 bytes to assignment-02-24300980021-刘俊彦.html
